In [1]:
import pandas as pd
import numpy as np
import random
import os
from itertools import combinations, product
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Patch

Computing model overlap

Functions and code to test different model overlap through permutation testing

In [2]:
def test_nway_sign_overlap(
    feature_files, k, n_permutations=10000, n_rois=268,
    output_dir="./overlap_results"
):
    os.makedirs(output_dir, exist_ok=True)

    # -----------------------
    # STEP 1: Load signed features per model
    # -----------------------
    feature_sets = {}
    for name, path in feature_files.items():
        df = pd.read_csv(path)

        signed_features = {}
        for _, row in df.iterrows():
            edge = tuple(sorted((row['ROI1'], row['ROI2'])))
            sign = 'pos' if row['feature_weight'] > 0 else 'neg' if row['feature_weight'] < 0 else None
            if sign:
                signed_features[edge] = sign
        feature_sets[name] = signed_features

    all_possible_edges = list(combinations(range(n_rois), 2))
    results = []

    if k == 1:
        print("\n🧠 Feature summary by model (k=1):")
        summary_rows = []
        
        for model, features in feature_sets.items():
            pos_edges = [e for e, s in features.items() if s == "pos"]
            neg_edges = [e for e, s in features.items() if s == "neg"]
    
            print(f"\n📘 Model: {model}")
            print(f"  ➕ Positive features: {len(pos_edges)}")
            print(f"  ➖ Negative features: {len(neg_edges)}")
    
            # Optionally print top few features
            print("  Top 5 positive edges:", pos_edges[:5])
            print("  Top 5 negative edges:", neg_edges[:5])
    
            summary_rows.append({
                "Model": model,
                "Positive_Features": len(pos_edges),
                "Negative_Features": len(neg_edges),
                "Total_Features": len(pos_edges) + len(neg_edges)
            })
    
        summary_df = pd.DataFrame(summary_rows)
        summary_path = os.path.join(output_dir, "feature_sign_summary_k1.csv")
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        summary_df.to_csv(summary_path, index=False)
    
        print(f"\n✅ Saved k=1 summary to: {summary_path}")
        return summary_df


    # -----------------------
    # STEP 2: Loop over model combinations
    # -----------------------
    model_names = list(feature_sets.keys())
    for combo in combinations(model_names, k):
        combo_name = "_".join(combo)
        print(f"\n🔍 Testing overlap for model combo: {combo_name}")

        # -----------------------
        # STEP 3: Compute observed overlap
        # -----------------------
        edge_sign_map_list = [feature_sets[model] for model in combo]

        # Find edges that are in all models
        common_edges = set.intersection(*[set(d.keys()) for d in edge_sign_map_list])

        # Keep only those where all signs match
        consistent_edges = [
            edge for edge in common_edges
            if len(set(d[edge] for d in edge_sign_map_list)) == 1
        ]

        observed_overlap = len(consistent_edges)
        print(f"Observed overlap (sign-consistent): {observed_overlap}")

        # Save observed overlap
        # Get the sign from any model (they all have the same sign for this edge)
        signs = []
        for edge in consistent_edges:
            # All models in the combo have same sign, so we can use the first one
            sign = feature_sets[combo[0]][edge]
            signs.append(sign)
        
        # Save as DataFrame with sign column
        # Save as DataFrame with sign and lobe names
        overlap_df = pd.DataFrame(consistent_edges, columns=["ROI1", "ROI2"])
        overlap_df["Sign"] = signs
        overlap_df["ROI1_Name"] = overlap_df["ROI1"].map(roi_to_lobe)
        overlap_df["ROI2_Name"] = overlap_df["ROI2"].map(roi_to_lobe)
        
        # Write to CSV
        overlap_df.to_csv(os.path.join(output_dir, f"overlapping_features_{combo_name}.csv"), index=False)

        

        # -----------------------
        # STEP 4: Null distribution
        # -----------------------
        # Determine how many pos and neg features each model had
        model_sign_counts = {}
        for model in combo:
            signs = list(feature_sets[model].values())
            model_sign_counts[model] = {
                'pos': signs.count('pos'),
                'neg': signs.count('neg')
            }

        null_overlaps = []
        for _ in range(n_permutations):
            rand_sign_maps = []

            for model in combo:
                n_pos = model_sign_counts[model]['pos']
                n_neg = model_sign_counts[model]['neg']
                n_total = n_pos + n_neg

                sampled_edges = random.sample(all_possible_edges, n_total)
                sampled_pos = set(sampled_edges[:n_pos])
                sampled_neg = set(sampled_edges[n_pos:])

                sign_map = {}
                for edge in sampled_pos:
                    sign_map[tuple(sorted(edge))] = 'pos'
                for edge in sampled_neg:
                    sign_map[tuple(sorted(edge))] = 'neg'

                rand_sign_maps.append(sign_map)

            # Compute sign-consistent intersection
            common_edges = set.intersection(*[set(d.keys()) for d in rand_sign_maps])
            consistent_null_edges = [
                edge for edge in common_edges
                if len(set(d[edge] for d in rand_sign_maps)) == 1
            ]

            null_overlaps.append(len(consistent_null_edges))

        null_overlaps = np.array(null_overlaps)
        p_value = (1 + np.sum(null_overlaps >= observed_overlap)) / (n_permutations + 1)
        print(f"Empirical p-value: {p_value:.6f}")

        # Save null
        np.save(os.path.join(output_dir, f"null_dist_{combo_name}.npy"), null_overlaps)

        # Record
        results.append({
            "Combination": combo_name,
            "Set_Size": k,
            "Observed_Overlap": observed_overlap,
            "Mean_Null_Overlap": np.mean(null_overlaps),
            "P_Value": p_value
        })

    # -----------------------
    # STEP 5: Save summary
    # -----------------------
    result_df = pd.DataFrame(results)
    summary_path = os.path.join(output_dir, f"overlap_results_{k}way_by_model.csv")
    result_df.to_csv(summary_path, index=False)
    print(f"\n✅ Saved results to: {summary_path}")

    # Append to master file
    master_summary_path = os.path.join(output_dir, "master_overlap_summary.csv")
    if os.path.exists(master_summary_path):
        result_df.to_csv(master_summary_path, mode='a', header=False, index=False)
    else:
        result_df.to_csv(master_summary_path, index=False)
    print(f"📌 Appended to: {master_summary_path}")

    return result_df


In [4]:
shen_df = pd.read_csv("../3_atlases/Shen/shen_roi_labels.csv")
roi_to_lobe = shen_df.set_index("ROI")["BA Name"].to_dict()

feature_files = {
    "PupilRest": "../4_results/2_classification/4_EEG_Rest_from_Pupil_Rest/EEG_Rest_from_Pupil_Rest_features.csv",
    "EEGRest": "../4_results/2_classification/5_Pupil_Rest_from_EEG_Rest/Pupil_Rest_from_EEG_Rest_features.csv",
    "PupilStory": "../4_results/2_classification/7_Pupil_Rest_from_Pupil_Story/Pupil_Rest_from_Pupil_Story_features.csv",
    "Movies": "../4_results/2_classification/1_Movies_within/Movies_features.csv"
}

output_dir = "../4_results/3_network/"

results = test_nway_sign_overlap(feature_files, k=3, n_permutations=1000, n_rois=268, output_dir = output_dir)



🔍 Testing overlap for model combo: PupilRest_EEGRest_PupilStory
Observed overlap (sign-consistent): 46
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: PupilRest_EEGRest_Movies
Observed overlap (sign-consistent): 105
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: PupilRest_PupilStory_Movies
Observed overlap (sign-consistent): 151
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: EEGRest_PupilStory_Movies
Observed overlap (sign-consistent): 83
Empirical p-value: 0.000999

✅ Saved results to: ../4_results/3_network/overlap_results_3way_by_model.csv
📌 Appended to: ../4_results/3_network/master_overlap_summary.csv


In [5]:
results_2 = test_nway_sign_overlap(feature_files, k=2, n_permutations=1000, n_rois=268, output_dir = output_dir)



🔍 Testing overlap for model combo: PupilRest_EEGRest
Observed overlap (sign-consistent): 385
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: PupilRest_PupilStory
Observed overlap (sign-consistent): 457
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: PupilRest_Movies
Observed overlap (sign-consistent): 890
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: EEGRest_PupilStory
Observed overlap (sign-consistent): 224
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: EEGRest_Movies
Observed overlap (sign-consistent): 453
Empirical p-value: 0.000999

🔍 Testing overlap for model combo: PupilStory_Movies
Observed overlap (sign-consistent): 600
Empirical p-value: 0.000999

✅ Saved results to: ../4_results/3_network/overlap_results_2way_by_model.csv
📌 Appended to: ../4_results/3_network/master_overlap_summary.csv


In [6]:
results_4 = test_nway_sign_overlap(feature_files, k=4, n_permutations=1000, n_rois=268, output_dir = output_dir)



🔍 Testing overlap for model combo: PupilRest_EEGRest_PupilStory_Movies
Observed overlap (sign-consistent): 23
Empirical p-value: 0.000999

✅ Saved results to: ../4_results/3_network/overlap_results_4way_by_model.csv
📌 Appended to: ../4_results/3_network/master_overlap_summary.csv
